In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")

import sys, time, shutil, subprocess, random, threading, math
import numpy as np
import psutil
import tensorflow as tf
from contextlib import contextmanager

MODEL_PATH   = sys.argv[1] if len(sys.argv) > 1 else "vgg16_fmnist_9367_tta.keras"
USE_GPU      = True
VRAM_MB      = 2048
BATCH_SIZE   = 32
NORMALIZE01  = True
WARMUP_STEPS = 400
RUN_STEPS    = 200
CADENCE_MS   = 40.0
SAMPLER_HZ   = 10.0

In [ ]:
SEED = 1337
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
if hasattr(tf.config.experimental, "enable_op_determinism"):
    try: tf.config.experimental.enable_op_determinism()
    except TypeError: tf.config.experimental.enable_op_determinism(True)
tf.config.threading.set_intra_op_parallelism_threads(int(os.environ["TF_NUM_INTRAOP_THREADS"]))
tf.config.threading.set_inter_op_parallelism_threads(int(os.environ["TF_NUM_INTEROP_THREADS"]))

@contextmanager
def sync_scope():
    try:
        async_scope = tf.experimental.async_scope
    except AttributeError:
        async_scope = None
    if async_scope is not None:
        try:
            with async_scope(False):
                yield; return
        except TypeError:
            with async_scope():
                yield; return
    yield

def device_fence():
    if USE_GPU:
        with tf.device("/GPU:0"):
            _ = (tf.reduce_sum(tf.ones((1,), dtype=tf.float32))).numpy()

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if USE_GPU:
    if not gpus:
        raise RuntimeError("USE_GPU=True but no GPU detected.")
    exp = tf.config.experimental
    if hasattr(exp, "set_logical_device_configuration") and hasattr(tf.config, "LogicalDeviceConfiguration"):
        exp.set_visible_devices(gpus[0], 'GPU')
        exp.set_logical_device_configuration(
            gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=VRAM_MB)]
        )
        api = "logical"
    elif hasattr(exp, "set_virtual_device_configuration") and hasattr(exp, "VirtualDeviceConfiguration"):
        exp.set_visible_devices(gpus[0], 'GPU')
        exp.set_virtual_device_configuration(
            gpus[0], [exp.VirtualDeviceConfiguration(memory_limit=VRAM_MB)]
        )
        api = "virtual"
    else:
        raise RuntimeError("No device configuration API found to cap VRAM.")
    print(f"GPU mode with fixed VRAM cap = {VRAM_MB} MB ({api} API)")
else:
    try: tf.config.set_visible_devices([], 'GPU')
    except: pass
    print("CPU-only mode")

In [ ]:
print(f"Loading model from: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)
ishape = model.inputs[0].shape
if len(ishape) == 4:
    _, H, W, C = ishape
elif len(ishape) == 3:
    H, W, C = ishape
else:
    raise ValueError(f"Unsupported input rank: {ishape}")
H = int(H) if H else 48
W = int(W) if W else 48
C = int(C) if C else 3
print(f"Fixed compute shape: batch={BATCH_SIZE}, HWC=({H},{W},{C})")

def to_3ch(x):
    if x.ndim == 2:
        x = x[..., None]
    if x.shape[-1] == 1:
        x = np.repeat(x, 3, axis=-1)
    return x

def preprocess_np(batch_any):
    if isinstance(batch_any, list):
        batch_any = np.stack(batch_any, axis=0)
    if batch_any.ndim == 3:
        batch_any = batch_any[..., None]
    N = batch_any.shape[0]
    out = np.empty((N, H, W, C), dtype=np.float32)
    for i in range(N):
        x = to_3ch(batch_any[i])
        tx = tf.image.resize(tf.convert_to_tensor(x), [H, W], method='bilinear', antialias=True)
        fx = tf.cast(tx, tf.float32)
        if NORMALIZE01: fx = fx / 255.0
        out[i] = fx.numpy()
    return out

In [ ]:
device_buffer = tf.Variable(tf.zeros([BATCH_SIZE, H, W, C], dtype=tf.float32), trainable=False)

@tf.function(input_signature=[
    tf.TensorSpec(shape=[BATCH_SIZE, H, W, C], dtype=tf.float32),
    tf.TensorSpec(shape=[BATCH_SIZE], dtype=tf.bool)
])
def infer_const(x_full, valid_mask):
    y = model(x_full, training=False)
    return y, valid_mask

def prepare_batches(arr):
    """Build padded, preprocessed batches from arbitrary N input (arr uint8)."""
    x = preprocess_np(arr)
    N = x.shape[0]
    batches = []
    for start in range(0, N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, N)
        real = end - start
        chunk = np.zeros((BATCH_SIZE, H, W, C), dtype=np.float32)
        chunk[:real] = x[start:end]
        mask = np.zeros((BATCH_SIZE,), dtype=bool)
        mask[:real] = True
        batches.append((chunk, mask))
    return batches

def set_process_affinity_and_priority():
    try:
        p = psutil.Process(os.getpid())
        cpus = list(range(max(1, psutil.cpu_count(logical=True)//2)))
        p.cpu_affinity(cpus)
    except Exception:
        pass
    try:
        if os.name == "posix":
            os.nice(-10)
    except Exception:
        pass

set_process_affinity_and_priority()

def has_nvidia_smi(): return shutil.which("nvidia-smi") is not None

In [ ]:
class ResourceMonitor:
    def __init__(self, hz=10.0, gpu_index="0"):
        self.dt = 1.0/float(hz)
        self.gpu_index = gpu_index
        self._stop = threading.Event()
        self._thr = None
        self.proc = psutil.Process(os.getpid())
        self.num_cpus = psutil.cpu_count(logical=True)

        self.tstamps = []
        self.gpu_util = []
        self.vram_used = []
        self.vram_total = []
        self.cpu_proc_pct = []
        self.rss_mb = []
        self.sys_ram_pct = []

        _ = self.proc.cpu_percent(interval=None)

    def _sample_gpu(self):
        if not has_nvidia_smi() or not USE_GPU: return (None, None, None)
        try:
            out = subprocess.check_output(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total",
                 "--format=csv,noheader,nounits","-i", self.gpu_index],
                stderr=subprocess.STDOUT
            ).decode().strip()
            util, used, total = [float(s.strip()) for s in out.split(",")]
            return util, used, total
        except Exception:
            return (None, None, None)

    def _loop(self):
        while not self._stop.is_set():
            t = time.perf_counter()
            util, used, total = self._sample_gpu()
            self.tstamps.append(t)
            if util is not None: self.gpu_util.append(util)
            if used is not None: self.vram_used.append(used)
            if total is not None: self.vram_total.append(total)

            self.cpu_proc_pct.append(self.proc.cpu_percent(interval=None))
            self.rss_mb.append(self.proc.memory_info().rss / (1024*1024))
            self.sys_ram_pct.append(psutil.virtual_memory().percent)

            self._stop.wait(self.dt)

    def start(self):
        self._stop.clear()
        self._thr = threading.Thread(target=self._loop, daemon=True)
        self._thr.start()

    def stop(self):
        self._stop.set()
        if self._thr: self._thr.join(timeout=1.0)

    @staticmethod
    def _brief(a):
        if not a: return (0.0, 0.0, 0.0, 0.0)
        arr = np.asarray(a, dtype=float)
        return (float(np.mean(arr)), float(np.std(arr)), float(np.median(arr)), float(np.percentile(arr, 90)))

    def summary(self):
        return {
            "gpu_util":   self._brief(self.gpu_util),
            "vram_used":  self._brief(self.vram_used),
            "rss_mb":     self._brief(self.rss_mb),
            "cpu_proc":   self._brief(self.cpu_proc_pct),
            "sys_ram":    self._brief(self.sys_ram_pct),
            "vram_total": (self.vram_total[-1] if self.vram_total else 0.0)
        }

In [ ]:
def keep_gpu_awake(seconds=1.5):
    t_end = time.perf_counter() + seconds
    with sync_scope():
        while time.perf_counter() < t_end:
            with tf.device("/GPU:0"):
                _ = tf.reduce_sum(tf.ones((1024, 1024), dtype=tf.float32))
        device_fence()

def run_fixed_cadence(batches, steps, cadence_ms):
    """Run 'steps' fixed-size inferences at fixed cadence (ms/step)."""
    period = cadence_ms / 1000.0
    lat_step_ms = []
    with sync_scope():
        for i in range(steps):
            chunk, mask = batches[i % len(batches)]
            device_buffer.assign(chunk)
            t0 = time.perf_counter()
            y, valid = infer_const(device_buffer, tf.convert_to_tensor(mask))
            device_fence()
            t1 = time.perf_counter()
            lat_ms = (t1 - t0) * 1000.0
            lat_step_ms.append(lat_ms)
            next_t = t0 + period
            now = time.perf_counter()
            rem = next_t - now
            if rem > 0:
                if rem > 0.002:
                    time.sleep(rem - 0.001)
                while time.perf_counter() < next_t:
                    pass
    return lat_step_ms

def brief(a):
    if not a: return (0.0, 0.0, 0.0, 0.0)
    arr = np.asarray(a, dtype=float)
    return (float(np.mean(arr)), float(np.std(arr)), float(np.median(arr)), float(np.percentile(arr, 90)))

In [ ]:
if __name__ == "__main__":
    N_FOR_POOL = BATCH_SIZE * 4
    raw = np.random.randint(0, 255, size=(N_FOR_POOL, H, W, C), dtype=np.uint8)
    batches = prepare_batches(raw)

    print("Deep warm-up ...")
    for _ in range(WARMUP_STEPS):
        chunk, mask = batches[_ % len(batches)]
        device_buffer.assign(chunk)
        with sync_scope():
            y, valid = infer_const(device_buffer, tf.convert_to_tensor(mask))
        if _ % 50 == 0:
            device_fence()
    device_fence()
    keep_gpu_awake(2.0)

    mon = ResourceMonitor(hz=SAMPLER_HZ)
    mon.start()

    print(f"Running {RUN_STEPS} steps at ~{CADENCE_MS:.1f} ms cadence ...")
    lat_ms = run_fixed_cadence(batches, RUN_STEPS, CADENCE_MS)

    mon.stop()
    summ = mon.summary()

    m, s, med, p90 = brief(lat_ms)
    print("\n----- Latency per step (inference only) -----")
    print(f"mean {m:.2f} ms | std {s:.2f} | median {med:.2f} | p90 {p90:.2f}")

    print("\n----- Resource usage (averaged over run) -----")
    def fmt4(tup): return f"mean {tup[0]:.1f} | std {tup[1]:.1f} | median {tup[2]:.1f} | p90 {tup[3]:.1f}"
    print(f"GPU util % : {fmt4(summ['gpu_util'])}")
    print(f"VRAM used MB: {fmt4(summ['vram_used'])} / total {summ['vram_total']:.0f} MB")
    print(f"CPU(proc) %: {fmt4(summ['cpu_proc'])}")
    print(f"RSS (MB)  : {fmt4(summ['rss_mb'])}")
    print(f"Sys RAM % : {fmt4(summ['sys_ram'])}")

    print("\nNotes:")
    print("- Cadenced stepping makes GPU/CPU usage steady by pacing to a fixed period.")
    print("- Metrics are sampled at steady 10 Hz and summarized over the entire run.")
    print("- VRAM stays constant via logical/virtual device memory limit + preallocated buffers.")
    print("- For even tighter stability: AC power + NVIDIA 'Prefer Maximum Performance' + close background apps.")